In [1]:
# Import packages
import pandas as pd
import numpy as np

In [2]:
# Import data
all_states = pd.read_parquet("../data/all_states.parquet")

# convert numbers to numeric
numeric_cols = ['StudentGroup_TotalTested', 'AvgScaleScore', 'ProficientOrAbove_count', 'ProficientOrAbove_percent']
for col in numeric_cols:
    all_states[col] = pd.to_numeric(all_states[col], errors='coerce')


In [3]:

# select only needed columns for district percentiles
selected_columns = ['StateAbbrev', 'SchYear', 'DataLevel', 'DistName', 'NCESDistrictID', 
                   'AssmtName', 'AssmtType', 'Subject', 'GradeLevel', 'StudentGroup_TotalTested',
                   'AvgScaleScore', 'ProficientOrAbove_count', 'ProficientOrAbove_percent', 'ParticipationRate',
                   'DistType', 'DistCharter', 'CountyName','CountyCode']

analysis = all_states[selected_columns]

In [4]:
# select needed rows
analysis = analysis[(analysis['DataLevel'] == 'District') &
                    (analysis['Subject'] != 'sci')]

In [ ]:
# create percentiles by state, school year, subject, gradelevel

# define grouping columns
group_cols = ['StateAbbrev', 'SchYear', 'Subject', 'GradeLevel']

# PERCENTILE BASED ON SCALED SCORE

# calculate the rank within each group
analysis['rank_ss'] = (
    analysis.groupby(group_cols)['AvgScaleScore']
    .rank(method = 'min', ascending = True)
    .astype('Int64')
)

# calculate the count of each group
analysis['count_ss'] = (
    analysis.groupby(group_cols)['AvgScaleScore']
    .transform('count')
)

# calcualte percentile, ranging from 0 to 100 (or below 100 if ties at the top)
analysis['pctl_ss'] = ((analysis['rank_ss'] - 1) / (analysis['count_ss'] - 1) * 100).round(1)

# PERCENTILE BASED ON PERCENT PROFICIENT

# calculate the rank within each group
analysis['rank_pp'] = (
    analysis.groupby(group_cols)['ProficientOrAbove_percent']
    .rank(method = 'min', ascending = True)
    .astype('Int64')
)

# calculate the count of each group
analysis['count_pp'] = (
    analysis.groupby(group_cols)['ProficientOrAbove_percent']
    .transform('count')
)

# calcualte percentile, ranging from 0 to 100 (or below 100 if ties at the top)
analysis['pctl_pp'] = ((analysis['rank_pp'] - 1) / (analysis['count_pp'] - 1) * 100).round(1)

# to do
# figure out steps to get this online (see AI conversation)



In [ ]:
# Calculate percentiles using a loop (COMMENTED OUT FOR NOW BECAUSE MORE COMPLEX, BUT CONCISE)

# group_cols = ['StateAbbrev', 'SchYear', 'Subject', 'GradeLevel']
# metrics = {'ss': 'AvgScaleScore', 'pp': 'ProficientOrAbove_percent'}

# for suf, col in metrics.items():
#     grouped = analysis.groupby(group_cols)[col]
#     analysis[f'rank_{suf}'] = grouped.rank(method='min').astype('Int64')
#     analysis[f'count_{suf}'] = grouped.transform('count')
    
#     # Calculate percentile using the newly created intermediate columns
#     analysis[f'pctl_{suf}'] = ((analysis[f'rank_{suf}'] - 1) / (analysis[f'count_{suf}'] - 1) * 100).round(1)

In [8]:
# Save data locally

analysis.to_parquet('../data/test_percentiles.parquet', index=False)